In [1]:
import os
import sys
from pathlib import Path

import yaml
from pyspark.sql import functions as F

# Find project root automatically
project_root = Path.cwd()
while not (project_root / "configs").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

print("Project root:", project_root)
os.chdir(project_root)

Project root: /home/ubuntu/renewable-energy-forecasting-pipeline


In [2]:
PROJECT_USER_CONFIG = os.environ.get("PROJECT_USER_CONFIG")

with open(project_root / PROJECT_USER_CONFIG, "r") as f:
    user_config = yaml.safe_load(f)

bucket = user_config["aws"]["project_bucket"]
gold_prefix = user_config["aws"]["gold_prefix"]

gold_root = f"s3a://{bucket}/{gold_prefix}"
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("layer7_local_analysis")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.default.parallelism", "8")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.InstanceProfileCredentialsProvider"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ubuntu/.ivy2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6be88c80-7d45-451a-806e-ca60c1860f4c;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 408ms :: artifacts dl 23ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	--------------------------------

In [3]:
region_monthly_path = f"{gold_root}/region/monthly"

region_monthly = spark.read.parquet(region_monthly_path)

print("Loaded region_monthly")
region_monthly.printSchema()

26/04/28 09:07:21 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Loaded region_monthly
root
 |-- month: integer (nullable = true)
 |-- monthly_region_capacity_factor: double (nullable = true)
 |-- monthly_mean_wind_speed_ms: double (nullable = true)
 |-- min_daily_region_capacity_factor: double (nullable = true)
 |-- max_daily_region_capacity_factor: double (nullable = true)
 |-- daily_observation_count: long (nullable = true)
 |-- avg_station_count: double (nullable = true)
 |-- is_valid_monthly_region_index: boolean (nullable = true)
 |-- wind_power_class: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- state: string (nullable = true)



In [4]:
from src.storage.table_builders import build_gold_monthly_state_wind

In [5]:
gold_monthly_state = build_gold_monthly_state_wind(region_monthly)

In [ ]:
# row count
print("rows:", gold_monthly_state.count())

# grain check
from pyspark.sql import functions as F
gold_monthly_state.select(
    F.count("*").alias("total"),
    F.countDistinct("state", "year", "month").alias("distinct")
).show()

gold_monthly_state.select(
    F.min("monthly_region_capacity_factor"),
    F.max("monthly_region_capacity_factor")
).show()

# season distribution
gold_monthly_state.groupBy("season").count().show()

rows: 17664


+-----+--------+
|total|distinct|
+-----+--------+
|17664|   17664|
+-----+--------+



+-----------------------------------+-----------------------------------+
|min(monthly_region_capacity_factor)|max(monthly_region_capacity_factor)|
+-----------------------------------+-----------------------------------+
|                           0.001662|                           0.274203|
+-----------------------------------+-----------------------------------+



+------+-----+
|season|count|
+------+-----+
|winter| 4416|
|  fall| 4320|
|spring| 4464|
|summer| 4464|
+------+-----+



In [6]:
from pyspark.sql import functions as F
gold_monthly_state.select(
    F.count("*").alias("total"),
    F.countDistinct("state", "year", "month").alias("distinct")
)

gold_monthly_state.select(
    F.min("monthly_region_capacity_factor"),
    F.max("monthly_region_capacity_factor")
)

DataFrame[min(monthly_region_capacity_factor): double, max(monthly_region_capacity_factor): double]

In [ ]:
from src.storage.write_gold import write_gold_table

monthly_state_path = f"{gold_root}/analytics/monthly_state"

monthly_state_small = gold_monthly_state.coalesce(1)

write_gold_table(
    monthly_state_small,
    output_path=monthly_state_path,
    partition_cols=["year", "state"],
    mode="overwrite",
)

print("Wrote gold_monthly_state_wind to:", monthly_state_path)

Wrote gold_monthly_state_wind to: s3a://syed-datsbd-s2026/gold/wind/analytics/monthly_state


In [ ]:
monthly_state_readback = spark.read.parquet(monthly_state_path)

monthly_state_readback.select(
    F.count("*").alias("rows"),
    F.countDistinct("state", "year", "month").alias("distinct_grain"),
).show()

monthly_state_readback.printSchema()

+-----+--------------+
| rows|distinct_grain|
+-----+--------------+
|17664|         17664|
+-----+--------------+

root
 |-- month: integer (nullable = true)
 |-- season: string (nullable = true)
 |-- monthly_region_capacity_factor: double (nullable = true)
 |-- monthly_mean_wind_speed_ms: double (nullable = true)
 |-- min_daily_region_capacity_factor: double (nullable = true)
 |-- max_daily_region_capacity_factor: double (nullable = true)
 |-- capacity_factor_range: double (nullable = true)
 |-- daily_observation_count: long (nullable = true)
 |-- avg_station_count: double (nullable = true)
 |-- is_valid_monthly_region_index: boolean (nullable = true)
 |-- wind_power_class: integer (nullable = true)
 |-- is_high_wind_month: boolean (nullable = true)
 |-- is_low_wind_month: boolean (nullable = true)
 |-- year: integer (nullable = true)
 |-- state: string (nullable = true)



In [8]:
from src.storage.write_gold import write_gold_table

monthly_state_path = f"{gold_root}/analytics/monthly_state"
monthly_state_readback = spark.read.parquet(monthly_state_path)

monthly_state_readback.select(
    F.count("*").alias("rows"),
    F.countDistinct("state", "year", "month").alias("distinct_grain"),
)

DataFrame[rows: bigint, distinct_grain: bigint]

In [9]:
import importlib
import src.storage.table_builders as table_builders

importlib.reload(table_builders)

from src.storage.table_builders import build_gold_daily_region_wind

In [ ]:
from src.storage.table_builders import build_gold_daily_region_wind

region_daily_path = f"{gold_root}/region/daily"
region_daily = spark.read.parquet(region_daily_path)

gold_daily_region = build_gold_daily_region_wind(region_daily)

gold_daily_region.select(
    F.count("*").alias("rows"),
    F.countDistinct("state", "date_utc").alias("distinct_grain"),
).show()

gold_daily_region.select(
    F.min("daily_region_capacity_factor"),
    F.max("daily_region_capacity_factor"),
).show()

gold_daily_region.groupBy("season").count().show()

+------+--------------+
|  rows|distinct_grain|
+------+--------------+
|537449|        537449|
+------+--------------+



+---------------------------------+---------------------------------+
|min(daily_region_capacity_factor)|max(daily_region_capacity_factor)|
+---------------------------------+---------------------------------+
|                              0.0|                         0.900287|
+---------------------------------+---------------------------------+



+------+------+
|season| count|
+------+------+
|winter|132815|
|  fall|131040|
|spring|136895|
|summer|136699|
+------+------+



In [10]:
from src.storage.table_builders import build_gold_daily_region_wind

region_daily_path = f"{gold_root}/region/daily"
region_daily = spark.read.parquet(region_daily_path)

gold_daily_region = build_gold_daily_region_wind(region_daily)

gold_daily_region.select(
    F.count("*").alias("rows"),
    F.countDistinct("state", "date_utc").alias("distinct_grain"),
)

gold_daily_region.select(
    F.min("daily_region_capacity_factor"),
    F.max("daily_region_capacity_factor"),
)

DataFrame[min(daily_region_capacity_factor): double, max(daily_region_capacity_factor): double]

In [ ]:
from src.storage.write_gold import write_gold_table

daily_region_path = f"{gold_root}/analytics/daily_region"

daily_region_small = gold_daily_region.coalesce(8)

write_gold_table(
    daily_region_small,
    output_path=daily_region_path,
    partition_cols=["year", "state"],
    mode="overwrite",
)

print("Wrote gold_daily_region_wind to:", daily_region_path)

Wrote gold_daily_region_wind to: s3a://syed-datsbd-s2026/gold/wind/analytics/daily_region


In [ ]:
daily_region_readback = spark.read.parquet(daily_region_path)

daily_region_readback.select(
    F.count("*").alias("rows"),
    F.countDistinct("state", "date_utc").alias("distinct_grain"),
).show()

daily_region_readback.select(
    F.min("daily_region_capacity_factor"),
    F.max("daily_region_capacity_factor"),
).show()

+------+--------------+
|  rows|distinct_grain|
+------+--------------+
|537449|        537449|
+------+--------------+



+---------------------------------+---------------------------------+
|min(daily_region_capacity_factor)|max(daily_region_capacity_factor)|
+---------------------------------+---------------------------------+
|                              0.0|                         0.900287|
+---------------------------------+---------------------------------+



In [11]:
from src.storage.write_gold import write_gold_table

daily_region_path = f"{gold_root}/analytics/daily_region"

daily_region_readback = spark.read.parquet(daily_region_path)

daily_region_readback.select(
    F.count("*").alias("rows"),
    F.countDistinct("state", "date_utc").alias("distinct_grain"),
)

daily_region_readback.select(
    F.min("daily_region_capacity_factor"),
    F.max("daily_region_capacity_factor"),
)

DataFrame[min(daily_region_capacity_factor): double, max(daily_region_capacity_factor): double]

In [13]:
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
from src.storage.write_gold import write_gold_table

gold_extreme_events = gold_extreme_events.persist(StorageLevel.MEMORY_AND_DISK)

# materialize cache
gold_extreme_events.count()

# ---- Diagnostics (now fast) ----
gold_extreme_events.orderBy(F.desc("capacity_factor_z_score")).select(
    "state",
    "date_utc",
    "daily_region_capacity_factor",
    "state_avg_capacity_factor",
    "state_std_capacity_factor",
    "capacity_factor_z_score",
    "extreme_event_type",
).show(20, truncate=False)

gold_extreme_events.groupBy("extreme_event_type").count().show()

# ---- Write ----
extreme_events_path = f"{gold_root}/analytics/extreme_events"

write_gold_table(
    gold_extreme_events.coalesce(8),
    output_path=extreme_events_path,
    partition_cols=["year", "state"],
    mode="overwrite",
)

print("Wrote gold_extreme_event_windows to:", extreme_events_path)

# ---- Read-back validation ----
extreme_readback = spark.read.parquet(extreme_events_path)

extreme_readback.select(
    F.count("*").alias("rows"),
    F.countDistinct("state", "date_utc").alias("distinct_grain"),
).show()

extreme_readback.groupBy("extreme_event_type").count().show()

ERROR:root:KeyboardInterrupt while sending command.][Stage 157:>  (0 + 0) / 1]6]
Traceback (most recent call last):
  File "/home/ubuntu/renewable-energy-forecasting-pipeline/.venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ubuntu/renewable-energy-forecasting-pipeline/.venv/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
spark.stop()